In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import GHEtool as ghe
from scipy.optimize import curve_fit

import sys
sys.path.append(r"C:\Workdir\Develop\IOCS")
from borefield_modeling import Borefield

In [ ]:
## Generate borefield parameters

bor_params = dict()  # Dictionary to store the borefield parameters

# Set the following parameters equal to the parameters used in the Modelica model!!!!!!
bor_params["config_from_file"] = False # Boolean, if True, the borefield configuration is read from a file
bor_params["conf_file_path"] = "" # Path to the borefield configuration file
bor_params["N_1"] = 8 # Number of boreholes in the first direction
bor_params["N_2"] = 1 # Number of boreholes in the second direction
bor_params["nBor"] = bor_params["N_1"] * bor_params["N_2"]  # Total number of boreholes
bor_params["H"] = 125 # m, depth of the boreholes
bor_params["B"] = 6 # m, distance between boreholes
bor_params["D"] = 0.8  # m, buried depth of the boreholes
bor_params["r_b"] = 0.066  # m, radius of the boreholes
bor_params["mBorFie"] = 0.2*bor_params["nBor"] # kg/s, mass flow rate of the borefield
# Generate the borehole coordinates based on N_1, N_2, and B
coordinates = []
for i in range(bor_params["N_1"]):
    for j in range(bor_params["N_2"]):
        x = i * bor_params["B"]
        y = j * bor_params["B"]
        coordinates.append([x, y])
bor_coordinates = "["
for i in coordinates[:-1]:
    bor_coordinates += str(i[0]) + "," + str(i[1]) + "; "
bor_coordinates += str(coordinates[-1][0]) + "," + str(coordinates[-1][1]) + "]"
bor_params["bor_coordinates"] = bor_coordinates

# Load aggregation parameters
bor_params["lifetime"] = 40  # Lifetime of the borehole system (in years)
bor_params["timFin"] = bor_params["lifetime"] * 365 * 24 * 3600  # Maximum time for g-function calculation (in seconds)
# Parameters for load aggregation (short-term modelling)
bor_params["tLoaAgg"] = 3600  # Time resolution of load aggregation (in seconds)
bor_params["nCel"] = 1  # Number of cells of same size per level
bor_params["lvlBas"] = 2  # Base for growth between each level

# Parameters for long-term modelling (L3)
bor_params["tBorStep"] = 3600*730 # s, Total Time of one borefield time step in seconds
bor_params["nbSteps"] = 12*bor_params["lifetime"] # Total number of borefield steps over lifetime

# Fluid Temperature parameters
bor_params["T_cte"] = 10 # °C, constant temperature of the ground
bor_params["Tf_min"] = 0 # °C, minimum fluid temperature
bor_params["Tf_max"] = 17 # °C, maximum fluid temperature

# Ground Thermal properties
bor_params["cp_ground"] = 1400  # J/kg/K, specific heat capacity of the ground
bor_params["rho_ground"] = 900  # kg/m3, density of the ground
bor_params["k_ground"] = 1.9  # W/m/K, thermal conductivity of the ground
bor_params["volumetric_heat_capacity_ground"] = (bor_params["cp_ground"] * bor_params["rho_ground"])  # J/m3/K, volumetric heat capacity of the ground
bor_params["alpha_ground"] = bor_params["k_ground"] / (bor_params["rho_ground"] * bor_params["cp_ground"])  # m2/s, thermal diffusivity of the ground

# Peak durations
bor_params["peak_duration_ext"] = 24 # h, peak duration of extraction of the borefield
bor_params["peak_duration_inj"] = 24 # h, peak duration of injection of the borefield


# GHEtool parameters
bor_params["ground_data"] = ghe.GroundConstantTemperature(k_s=bor_params["k_ground"], T_g=bor_params["T_cte"], volumetric_heat_capacity= bor_params["volumetric_heat_capacity_ground"])
bor_params["pipe_data"] = ghe.DoubleUTube(k_g=2, r_in=0.0131, r_out=0.016, k_p=0.38, D_s=0.043)
bor_params["fluid_data"] = ghe.FluidData(mfr=bor_params["mBorFie"]/bor_params["nBor"], k_f=0.598, rho=995.586, Cp=4184, mu=1e-3)

In [ ]:
## Calculation parameters for the borefield
borfie = ghe.Borefield()
borfie.set_ground_parameters(bor_params["ground_data"])
borfie.set_fluid_parameters(bor_params["fluid_data"])
borfie.set_pipe_parameters(bor_params["pipe_data"])
borfie.calculation_setup(use_constant_Rb=False)

H_step = 5
min_depth = 50
max_depth = 300
H_range_linearisation = np.arange(min_depth, max_depth + H_step, H_step) # m, range of borehole depths for linearisation
bor_params["H_range_linearisation"] = H_range_linearisation

## Kappas for different depths
bor_params["kappas_L3"] = dict() # Kappa values for LT model
bor_params["kappas_agg"] = dict() # Kappa values for ST model
bor_params["kappa_L3_peak_ext"] = dict() # Peak extraction weighting factor
bor_params["kappa_L3_peak_inj"] = dict() # Peak injection weighting factor

# Calculate the aggregation weighting factors for different depths
for H in H_range_linearisation:
    bor_params["H"] = H # m, depth of the boreholes
    # Create borefield object
    bf = Borefield(bor_params)

    # Calculate the g-functions
    bf.calculate_g_function()
    timSer = np.column_stack(
                (bor_params["time_gfunc"], bor_params["Tstep_gfunc"])
            )
    
    # Aggregation model (short-term)
    nb_agg_cells = bf.countAggregationCells(bor_params["lvlBas"], bor_params["nCel"], bor_params["timFin"], bor_params["tLoaAgg"])
    nu, rCel_agg = bf.aggregationCellTimes(nb_agg_cells, bor_params["lvlBas"], bor_params["nCel"], bor_params["tLoaAgg"], bor_params["timFin"])
    bor_params["kappas_agg"][H] = bf.aggregationWeightingFactors(i=nb_agg_cells, 
                                                                 nTimTot=len(bor_params["Tstep_gfunc"]), 
                                                                TStep=timSer,
                                                                nu=nu)
                                                        

    ## LT model (L3)
    rCel = np.ones(bor_params["nbSteps"]) * bor_params["tBorStep"]
    nu = np.cumsum(rCel)
    bor_params["kappas_L3"][H] = bf.aggregationWeightingFactors(
                i=bor_params["nbSteps"],
                nTimTot=len(timSer),
                TStep=timSer,
                nu=nu,
            )
    
    peak_duration_ext = bor_params["peak_duration_ext"]
    peak_duration_inj = bor_params["peak_duration_inj"]
    nu_peak_ext = np.array([peak_duration_ext*3600])
    nu_peak_inj = np.array([peak_duration_inj*3600])
    bor_params["kappa_L3_peak_ext"][H] = bf.aggregationWeightingFactors(
                    i=1, nTimTot=len(timSer), TStep=timSer, nu=nu_peak_ext)
    bor_params["kappa_L3_peak_inj"][H] = bf.aggregationWeightingFactors(
                    i=1, nTimTot=len(timSer), TStep=timSer, nu=nu_peak_inj)
    

# Monthly fluid temperature correction factors
Tf_corr_monthly = np.zeros(len(H_range_linearisation))
Tf_corr_peak_ext = np.zeros(len(H_range_linearisation))
Tf_corr_peak_inj = np.zeros(len(H_range_linearisation))
for i, H in enumerate(H_range_linearisation):
    Rb = borfie.borehole.get_Rb(H=H, D=bor_params["D"], r_b=bor_params["r_b"], k_s=bor_params["k_ground"])
    Tf_corr_monthly[i] = Rb/(bor_params["nBor"]*H)
    Tf_corr_peak_ext[i] = bor_params["kappa_L3_peak_ext"][H][0] + Rb/(bor_params["nBor"]*H)
    Tf_corr_peak_inj[i] = bor_params["kappa_L3_peak_inj"][H][0] + Rb/(bor_params["nBor"]*H)

bor_params["Tf_corr_monthly"] = Tf_corr_monthly
bor_params["Tf_corr_peak_ext"] = Tf_corr_peak_ext
bor_params["Tf_corr_peak_inj"] = Tf_corr_peak_inj

In [ ]:
## define the functions for curve fitting
# Define the linear function
def linear(H, a, b):
    return a * H + b

# Define the quadratic function
def quadratic(H, a, b, c):
    return a * H**2 + b * H + c

# Define the cubic function
def cubic(H, a, b, c, d):
    return a * H**3 + b * H**2 + c * H + d

# Linear rational function: (a*H + b) / (c*H + d)
def rational_linear(H, a, b, c, d):
    return (a * H + b) / (c * H + d)

# Quadratic rational function: (a*H**2 + b*H + c) / (d*H + e)
def rational_quadratic(H, a, b, c, d, e):
    return (a * H**2 + b * H + c) / (d * H + e)

# Cubic rational function: (a*H**3 + b*H**2 + c*H + d) / (e*H + f)
def rational_cubic(H, a, b, c, d, e, f):
    return (a * H**3 + b * H**2 + c * H + d) / (e * H + f)

# 4th rational function: (a*H**4 + b*H**3 + c*H**2 + d*H + e) / (f*H + g)
def rational_quartic(H, a, b, c, d, e, f, g):
    return (a * H**4 + b * H**3 + c * H**2 + d * H + e) / (f * H + g)

def r2_score(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - ss_res / ss_tot if ss_tot != 0 else np.nan

def find_best_estimator(xdata, ydata):
    models = {
        'linear': linear,
        'quadratic': quadratic,
        'cubic': cubic,
        'rational_linear': rational_linear,
        'rational_quadratic': rational_quadratic,
        'rational_cubic': rational_cubic,
        'rational_quartic': rational_quartic
    }


    def get_equation_and_constants(name, popt):
        # Returns a string representation of the fitted equation
        # equation format: (A*H^4 + B*H^3 + C*H^2 + D*H + E)/(F*H + G)
        A = B = C = D = E = F = G = 0.0
        G = 1 # Default value for G to avoid division by zero
        if name == 'linear':
            D, E = popt[0], popt[1]
            eq = f"{popt[0]:.14g}*H+{popt[1]:.14g}".replace('+-', '-')

        elif name == 'quadratic':
            C, D, E = popt[0], popt[1], popt[2]
            eq = f"{popt[0]:.14g}*H**2+{popt[1]:.14g}*H+{popt[2]:.14g}".replace('+-', '-')
        elif name == 'cubic':
            B, C, D, E = popt[0], popt[1], popt[2], popt[3]
            eq = f"{popt[0]:.14g}*H**3+{popt[1]:.14g}*H**2+{popt[2]:.14g}*H+{popt[3]:.14g}".replace('+-', '-')
        elif name == 'rational_linear':
            D, E, F, G = popt[0], popt[1], popt[2], popt[3]
            eq = f"({popt[0]:.14g}*H+{popt[1]:.14g})/({popt[2]:.14g}*H+{popt[3]:.14g})".replace('+-', '-')
        elif name == 'rational_quadratic':
            C, D, E, F, G = popt[0], popt[1], popt[2], popt[3], popt[4]
            eq = f"({popt[0]:.14g}*H**2+{popt[1]:.14g}*H+{popt[2]:.14g})/({popt[3]:.14g}*H+{popt[4]:.14g})".replace('+-', '-')
        elif name == 'rational_cubic':
            B, C, D, E, F, G = popt[0], popt[1], popt[2], popt[3], popt[4], popt[5]
            eq = f"({popt[0]:.14g}*H**3+{popt[1]:.14g}*H**2+{popt[2]:.14g}*H+{popt[3]:.14g})/({popt[4]:.14g}*H+{popt[5]:.14g})".replace('+-', '-')
        elif name == 'rational_quartic':
            A, B, C, D, E, F, G = popt[0], popt[1], popt[2], popt[3], popt[4], popt[5], popt[6]
            eq = f"({popt[0]:.14g}*H**4+{popt[1]:.14g}*H**3+{popt[2]:.14g}*H**2+{popt[3]:.14g}*H+{popt[4]:.14g})/({popt[5]:.14g}*H+{popt[6]:.14g})".replace('+-', '-')
        else:
            eq = "N/A"
        Constants = (A, B, C, D, E, F, G)

        return eq, Constants

    best_model = None
    best_score = -np.inf
    best_equation = None

    for name, model in models.items():
        try:
            popt, _ = curve_fit(model, xdata, ydata,  maxfev=10000)
            y_pred = model(xdata, *popt)
            score = r2_score(ydata, y_pred)
            if score > best_score:
                best_score = score
                best_model = (name, popt)
                best_equation, best_constants = get_equation_and_constants(name, popt)
        except Exception as e:
            print(f"Error fitting model {name}: {e}")
    print("best_model: ", best_model[0], best_constants)
    
    return best_model, best_equation, best_constants

In [ ]:
## Curve fitting

## Aggregation model (short-term)
best_params_kappas_agg = []
best_constants_kappas_agg = []
best_func_type_kappas_agg = []
kappa_agg_equations = [None for _ in range(nb_agg_cells)]
for i in range(nb_agg_cells):
    ydata = np.array([bor_params["kappas_agg"][h][i] for h in H_range_linearisation])
    params, ftype, constants = find_best_estimator(H_range_linearisation, ydata)
    best_func_type_kappas_agg.append(ftype)
    best_params_kappas_agg.append(params)
    best_constants_kappas_agg.append(constants)
    kappa_agg_equations[i] = ftype

## L3 model
best_params_kappas_L3 = []
best_constants_kappas_L3 = []
best_func_type_kappas_L3 = []
kappa_L3_equations = [None for _ in range(bor_params["nbSteps"])]
for i in range(bor_params["nbSteps"]):
    ydata = np.array([bor_params["kappas_L3"][h][i] for h in H_range_linearisation])
    params, ftype, constants = find_best_estimator(H_range_linearisation, ydata)
    best_func_type_kappas_L3.append(ftype)
    best_params_kappas_L3.append(params)
    best_constants_kappas_L3.append(constants)
    kappa_L3_equations[i] = ftype

# Monthly
ydata = np.array([Tf_corr_monthly[i] for i in range(len(H_range_linearisation))])
best_model_monthly, Tf_corr_monthly_eq, Tf_corr_monthly_constants = find_best_estimator(H_range_linearisation, ydata)
print("Monthly best estimator:", best_model_monthly, Tf_corr_monthly_eq)

# Peak extraction
ydata = np.array([Tf_corr_peak_ext[i] for i in range(len(H_range_linearisation))])
best_model_peak_ext, Tf_corr_peak_ext_eq, Tf_corr_peak_ext_constants = find_best_estimator(H_range_linearisation, ydata)
print("Peak extraction best estimator:", best_model_peak_ext, Tf_corr_peak_ext_eq)

# Peak injection
ydata = np.array([Tf_corr_peak_inj[i] for i in range(len(H_range_linearisation))])
best_model_peak_inj, Tf_corr_peak_inj_eq, Tf_corr_peak_inj_constants = find_best_estimator(H_range_linearisation, ydata)
print("Peak injection best estimator:", best_model_peak_inj, Tf_corr_peak_inj_eq)

In [ ]:
print("rCel:")
print(f"{{{', '.join(map(str, rCel_agg))}}}")

In [ ]:
# Print aggregation constants
print(f"\tparameter Real Kappa_agg_Constants[{nb_agg_cells}, 7] = [")
for i in range(nb_agg_cells):
    if i == nb_agg_cells-1:
        print(f"\t\t{best_constants_kappas_agg[i][0]}, {best_constants_kappas_agg[i][1]}, {best_constants_kappas_agg[i][2]}, {best_constants_kappas_agg[i][3]}, {best_constants_kappas_agg[i][4]}, {best_constants_kappas_agg[i][5]}, {best_constants_kappas_agg[i][6]}")
    else:
        print(f"\t\t{best_constants_kappas_agg[i][0]}, {best_constants_kappas_agg[i][1]}, {best_constants_kappas_agg[i][2]}, {best_constants_kappas_agg[i][3]}, {best_constants_kappas_agg[i][4]}, {best_constants_kappas_agg[i][5]}, {best_constants_kappas_agg[i][6]};")
print("\t];")

# Print L3 constants
print(f"\tparameter Real Gamma_Constants[7] = {{{Tf_corr_monthly_constants[0]}, {Tf_corr_monthly_constants[1]}, {Tf_corr_monthly_constants[2]}, {Tf_corr_monthly_constants[3]}, {Tf_corr_monthly_constants[4]}, {Tf_corr_monthly_constants[5]}, {Tf_corr_monthly_constants[6]}}};")
print(f"\tparameter Real Xi_Ext_Constants[7] = {{{Tf_corr_peak_ext_constants[0]}, {Tf_corr_peak_ext_constants[1]}, {Tf_corr_peak_ext_constants[2]}, {Tf_corr_peak_ext_constants[3]}, {Tf_corr_peak_ext_constants[4]}, {Tf_corr_peak_ext_constants[5]}, {Tf_corr_peak_ext_constants[6]}}};")
print(f"\tparameter Real Xi_Inj_Constants[7] = {{{Tf_corr_peak_inj_constants[0]}, {Tf_corr_peak_inj_constants[1]}, {Tf_corr_peak_inj_constants[2]}, {Tf_corr_peak_inj_constants[3]}, {Tf_corr_peak_inj_constants[4]}, {Tf_corr_peak_inj_constants[5]}, {Tf_corr_peak_inj_constants[6]}}};")
print(f"\tparameter Real Kappa_L3_Constants[{bor_params['nbSteps']}, 7] = [")
for i in range(bor_params["nbSteps"]):
    if i == bor_params["nbSteps"]-1:
        print(f"\t\t{best_constants_kappas_L3[i][0]}, {best_constants_kappas_L3[i][1]}, {best_constants_kappas_L3[i][2]}, {best_constants_kappas_L3[i][3]}, {best_constants_kappas_L3[i][4]}, {best_constants_kappas_L3[i][5]}, {best_constants_kappas_L3[i][6]}")
    else:
        print(f"\t\t{best_constants_kappas_L3[i][0]}, {best_constants_kappas_L3[i][1]}, {best_constants_kappas_L3[i][2]}, {best_constants_kappas_L3[i][3]}, {best_constants_kappas_L3[i][4]}, {best_constants_kappas_L3[i][5]}, {best_constants_kappas_L3[i][6]};")
print("\t];")



In [ ]:
%matplotlib qt
plt.close('all')
import pandas as pd

# Font size settings
title_fontsize = 18
label_fontsize = 16
tick_fontsize = 14
legend_fontsize = 14

def evaluate_H_equation(H, eq):
    """
    Evaluate the kappa value for a given borehole depth H using the provided kappa equation.
    """
    return eval(eq, {}, {"H": H})

H = bor_params["H_range_linearisation"]

def compute_errors(y_actual, y_est):
    abs_error = y_actual - y_est
    rel_error = np.where(y_actual != 0, abs_error / y_actual, 0)
    r2 = r2_score(y_actual, y_est)
    avg_rel_error = np.mean(np.abs(rel_error))
    return abs_error, rel_error, r2, avg_rel_error

# Plot for kappas_agg (every 2 steps)
for idx in range(0, nb_agg_cells, 2):
    y_actual = np.array([bor_params["kappas_agg"][h][idx] for h in H])
    eq = kappa_agg_equations[idx]
    y_est = np.array([evaluate_H_equation(h, eq) for h in H])
    abs_error, rel_error, r2, avg_rel_error = compute_errors(y_actual, y_est)
    fig, axs = plt.subplots(3, 1, figsize=(7, 10), sharex=True)
    axs[0].plot(H, y_actual, 'o', label='Actual')
    axs[0].plot(H, y_est, '-', label=f'Estimated\nR2={r2:.4f}, AvgRelErr={avg_rel_error:.2%}')
    axs[0].set_title(f'kappas_agg[{idx}]', fontsize=title_fontsize)
    axs[0].set_ylabel('Kappa', fontsize=label_fontsize)
    axs[0].legend(fontsize=legend_fontsize)
    axs[1].plot(H, abs_error, 'r-', label='Absolute Error')
    axs[1].set_ylabel('Absolute Error', fontsize=label_fontsize)
    axs[1].legend(fontsize=legend_fontsize)
    axs[2].plot(H, rel_error*100, 'g-', label='Relative Error')
    axs[2].set_xlabel('Borehole Depth H [m]', fontsize=label_fontsize)
    axs[2].set_ylabel('Relative Error [%]', fontsize=label_fontsize)
    axs[2].legend(fontsize=legend_fontsize)
    for ax in axs:
        ax.grid(True)
        ax.tick_params(axis='both', labelsize=tick_fontsize)
    plt.tight_layout()

# Plot for Gamma (Tf_corr_monthly)
y_actual = Tf_corr_monthly
y_est = np.array([evaluate_H_equation(h, Tf_corr_monthly_eq) for h in H])
abs_error, rel_error, r2, avg_rel_error = compute_errors(y_actual, y_est)
fig, axs = plt.subplots(3, 1, figsize=(7, 10), sharex=True)
axs[0].plot(H, y_actual, 'o', label='Actual')
axs[0].plot(H, y_est, '-', label=f'Estimated\nR2={r2:.4f}, AvgRelErr={avg_rel_error:.2%}')
axs[0].set_title('Gamma (Tf_corr_monthly)', fontsize=title_fontsize)
axs[0].set_ylabel('Gamma', fontsize=label_fontsize)
axs[0].legend(fontsize=legend_fontsize)
axs[1].plot(H, abs_error, 'r-', label='Absolute Error')
axs[1].set_ylabel('Absolute Error', fontsize=label_fontsize)
axs[1].legend(fontsize=legend_fontsize)
axs[2].plot(H, rel_error*100, 'g-', label='Relative Error')
axs[2].set_xlabel('Borehole Depth H [m]', fontsize=label_fontsize)
axs[2].set_ylabel('Relative Error [%]', fontsize=label_fontsize)
axs[2].legend(fontsize=legend_fontsize)
for ax in axs:
    ax.grid(True)
    ax.tick_params(axis='both', labelsize=tick_fontsize)
plt.tight_layout()

# Plot for Xi_Ext (Tf_corr_peak_ext)
y_actual = Tf_corr_peak_ext
y_est = np.array([evaluate_H_equation(h, Tf_corr_peak_ext_eq) for h in H])
abs_error, rel_error, r2, avg_rel_error = compute_errors(y_actual, y_est)
fig, axs = plt.subplots(3, 1, figsize=(7, 10), sharex=True)
axs[0].plot(H, y_actual, 'o', label='Actual')
axs[0].plot(H, y_est, '-', label=f'Estimated\nR2={r2:.4f}, AvgRelErr={avg_rel_error:.2%}')
axs[0].set_title('Xi_Ext (Tf_corr_peak_ext)', fontsize=title_fontsize)
axs[0].set_ylabel('Xi_Ext', fontsize=label_fontsize)
axs[0].legend(fontsize=legend_fontsize)
axs[1].plot(H, abs_error, 'r-', label='Absolute Error')
axs[1].set_ylabel('Absolute Error', fontsize=label_fontsize)
axs[1].legend(fontsize=legend_fontsize)
axs[2].plot(H, rel_error*100, 'g-', label='Relative Error')
axs[2].set_xlabel('Borehole Depth H [m]', fontsize=label_fontsize)
axs[2].set_ylabel('Relative Error [%]', fontsize=label_fontsize)
axs[2].legend(fontsize=legend_fontsize)
for ax in axs:
    ax.grid(True)
    ax.tick_params(axis='both', labelsize=tick_fontsize)
plt.tight_layout()

# Plot for Xi_Inj (Tf_corr_peak_inj)
y_actual = Tf_corr_peak_inj
y_est = np.array([evaluate_H_equation(h, Tf_corr_peak_inj_eq) for h in H])
abs_error, rel_error, r2, avg_rel_error = compute_errors(y_actual, y_est)
fig, axs = plt.subplots(3, 1, figsize=(7, 10), sharex=True)
axs[0].plot(H, y_actual, 'o', label='Actual')
axs[0].plot(H, y_est, '-', label=f'Estimated\nR2={r2:.4f}, AvgRelErr={avg_rel_error:.2%}')
axs[0].set_title('Xi_Inj (Tf_corr_peak_inj)', fontsize=title_fontsize)
axs[0].set_ylabel('Xi_Inj', fontsize=label_fontsize)
axs[0].legend(fontsize=legend_fontsize)
axs[1].plot(H, abs_error, 'r-', label='Absolute Error')
axs[1].set_ylabel('Absolute Error', fontsize=label_fontsize)
axs[1].legend(fontsize=legend_fontsize)
axs[2].plot(H, rel_error*100, 'g-', label='Relative Error')
axs[2].set_xlabel('Borehole Depth H [m]', fontsize=label_fontsize)
axs[2].set_ylabel('Relative Error [%]', fontsize=label_fontsize)
axs[2].legend(fontsize=legend_fontsize)
for ax in axs:
    ax.grid(True)
    ax.tick_params(axis='both', labelsize=tick_fontsize)
plt.tight_layout()

# Plot for kappas_L3 (every 50 steps)
for idx in range(0, bor_params["nbSteps"], 50):
    y_actual = np.array([bor_params["kappas_L3"][h][idx] for h in H])
    eq = kappa_L3_equations[idx]
    y_est = np.array([evaluate_H_equation(h, eq) for h in H])
    abs_error, rel_error, r2, avg_rel_error = compute_errors(y_actual, y_est)
    fig, axs = plt.subplots(3, 1, figsize=(7, 10), sharex=True)
    axs[0].plot(H, y_actual, 'o', label='Actual')
    axs[0].plot(H, y_est, '-', label=f'Estimated\nR2={r2:.4f}, AvgRelErr={avg_rel_error:.2%}')
    axs[0].set_title(f'kappas_L3[{idx}]', fontsize=title_fontsize)
    axs[0].set_ylabel('Kappa', fontsize=label_fontsize)
    axs[0].legend(fontsize=legend_fontsize)
    axs[1].plot(H, abs_error, 'r-', label='Absolute Error')
    axs[1].set_ylabel('Absolute Error', fontsize=label_fontsize)
    axs[1].legend(fontsize=legend_fontsize)
    axs[2].plot(H, rel_error*100, 'g-', label='Relative Error')
    axs[2].set_xlabel('Borehole Depth H [m]', fontsize=label_fontsize)
    axs[2].set_ylabel('Relative Error [%]', fontsize=label_fontsize)
    axs[2].legend(fontsize=legend_fontsize)
    for ax in axs:
        ax.grid(True)
        ax.tick_params(axis='both', labelsize=tick_fontsize)
    plt.tight_layout()
plt.show()


In [ ]:
# print summarizing table
summary_data = []

# kappas_agg (every 2 steps)
for idx in range(0, nb_agg_cells, 2):
    y_actual = np.array([bor_params["kappas_agg"][h][idx] for h in H])
    eq = kappa_agg_equations[idx]
    y_est = np.array([evaluate_H_equation(h, eq) for h in H])
    abs_error, rel_error, r2, avg_rel_error = compute_errors(y_actual, y_est)
    summary_data.append({
        "Name": f"kappas_agg[{idx}]",
        "R2": r2,
        "Avg Abs Error": np.mean(np.abs(abs_error)),
        "Avg Rel Error": avg_rel_error
    })

# Gamma (Tf_corr_monthly)
y_actual = Tf_corr_monthly
y_est = np.array([evaluate_H_equation(h, Tf_corr_monthly_eq) for h in H])
abs_error, rel_error, r2, avg_rel_error = compute_errors(y_actual, y_est)
summary_data.append({
    "Name": "Gamma (Tf_corr_monthly)",
    "R2": r2,
    "Avg Abs Error": np.mean(np.abs(abs_error)),
    "Avg Rel Error": avg_rel_error
})

# Xi_Ext (Tf_corr_peak_ext)
y_actual = Tf_corr_peak_ext
y_est = np.array([evaluate_H_equation(h, Tf_corr_peak_ext_eq) for h in H])
abs_error, rel_error, r2, avg_rel_error = compute_errors(y_actual, y_est)
summary_data.append({
    "Name": "Xi_Ext (Tf_corr_peak_ext)",
    "R2": r2,
    "Avg Abs Error": np.mean(np.abs(abs_error)),
    "Avg Rel Error": avg_rel_error
})

# Xi_Inj (Tf_corr_peak_inj)
y_actual = Tf_corr_peak_inj
y_est = np.array([evaluate_H_equation(h, Tf_corr_peak_inj_eq) for h in H])
abs_error, rel_error, r2, avg_rel_error = compute_errors(y_actual, y_est)
summary_data.append({
    "Name": "Xi_Inj (Tf_corr_peak_inj)",
    "R2": r2,
    "Avg Abs Error": np.mean(np.abs(abs_error)),
    "Avg Rel Error": avg_rel_error
})

# kappas_L3 (every 50 steps)
for idx in range(0, bor_params["nbSteps"], 50):
    y_actual = np.array([bor_params["kappas_L3"][h][idx] for h in H])
    eq = kappa_L3_equations[idx]
    y_est = np.array([evaluate_H_equation(h, eq) for h in H])
    abs_error, rel_error, r2, avg_rel_error = compute_errors(y_actual, y_est)
    summary_data.append({
        "Name": f"kappas_L3[{idx}]",
        "R2": r2,
        "Avg Abs Error": np.mean(np.abs(abs_error)),
        "Avg Rel Error": avg_rel_error
    })

summary_df = pd.DataFrame(summary_data)
print(summary_df)
